In [2]:
# A1 · Part 2 — Build a minimal retrieval system.  Starter notebook.
# PE6201 · Emerging AI Technologies. Run top to bottom, then replace the placeholder domain.

# A1 · Part 2 — Build a minimal retrieval system  *(starter)*

**What Part 2 asks of you.** Assemble a small collection of documents from **your** domain —
10 to 30 is enough. Build a system that retrieves relevant passages and uses them to answer
questions. Show it answering **five questions it could not possibly answer without your
documents**. Then **break it**: find a question where retrieval fails or returns the wrong
passage, and explain *why* it failed.

> **The breaking is not optional and it is not a penalty.** Finding the edge of your own system
> is the single most valuable thing you will do in this assignment. A notebook where everything
> works is a notebook that was not pushed hard enough.

**What this notebook is.** A working retrieval-augmented generation (RAG) pipeline on a
**placeholder domain** (houseplant care notes). Everything you must change is marked
**`### REPLACE ME`**.

**Cost.** About 10 short generation calls. Fractions of a cent. Embeddings run locally and free.

**Deliverable from this notebook:** this notebook (run, output visible), five worked examples,
one documented failure, and roughly 400 words.

> **If a cell does not work.** This notebook was written before you ran it, against library
> versions, model names and provider behaviour that change without notice — so treat it as a
> working draft, not gospel. **Read the error, then try to fix it yourself:** debugging is part of
> the work, not a detour from it. Then **report it** — in class or by email — with the cell, the
> full error text, and what you tried, so the shared copy gets fixed and nobody behind you loses
> the same hour. Reporting a problem costs you nothing. The one thing not to do is quietly work
> around a broken cell: if your fix changes what the notebook measures, say so in your writeup.

### Setup

In [3]:
%pip -q install openai scikit-learn sentence-transformers

^C
Note: you may need to restart the kernel to use updated packages.


### Connect to OpenRouter
The **generation** step (writing the answer) goes to OpenRouter with your key. The
**embedding** step (turning text into vectors) runs **locally and free** — OpenRouter serves chat
models, not embedding models, which is exactly the "which layer do you rent?" question from
Class 2.

In [1]:
import os, getpass, time
import numpy as np
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)
GEN_MODEL = 'openai/gpt-4o-mini'      # TRY THIS (knob): any model your key can reach
TOKENS = {'in': 0, 'out': 0}

def generate(prompt, system=None, max_new_tokens=300, temperature=0.0):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected ·', generate('Reply with exactly the word: ready', max_new_tokens=6))

connected · ready


### Check your remaining credit  —  free, this call is not billed
Your allocation is **fixed and will not be topped up**, so watch it rather than discovering it at
the end. `usage` is what you have spent; `limit` is your cap; the difference is what is left.
Run this at the start of a working session and again at the end.

In [2]:
import urllib.request, json as _json

def credit_check():
    """Ask OpenRouter what is left on your key. Costs nothing."""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.4901 of $10.00  ->  $9.5099 remaining
rate limit: -1 requests per 10s


---
## 0 · Five words you need — *start here if you don't code*

| Word | In plain words |
|---|---|
| **Embedding** | A piece of text turned into a list of numbers, arranged so that texts with similar *meaning* end up near each other. |
| **Chunk** | A small slice of a document. We search chunks, not whole documents, so what comes back is the relevant *paragraph* rather than the whole manual. |
| **Retrieval** | Finding the chunks nearest the question. Pure arithmetic — no model involved, no understanding. |
| **Top-k** | How many chunks we fetch. Too few and the answer is missing; too many and the real answer drowns in noise. |
| **Grounding** | Making the model answer **only** from the retrieved text, and say so when the text does not cover it. |

**RAG in one line:** *embed your documents → retrieve the ones nearest the question → let a model
answer from them.* **The model is rented; the retrieval is yours.** That is why retrieval is where
almost every failure lives — and why the failure you document is the interesting part.

---
## 1 · Your documents  ### REPLACE ME

**10 to 30 documents.** Each a short paragraph — a few sentences is fine.

**The one rule that matters:** they must contain things a model *could not already know*. If your
documents are summaries of Wikipedia, the model answers correctly with retrieval switched off, and
you have proved nothing. Good sources: your own notes, internal procedures, club or team records,
a rulebook nobody has digitised, your family's version of anything, recent local information,
specifics with names and numbers in them.

**Test for this later, in Section 4.** We ask the model each question with the documents removed.
If it still gets them right, your document set is too generic — go back and make it specific.

In [3]:
### REPLACE ME — your documents. 10-30 short paragraphs from YOUR domain.
DOCS = [
    "Letter of credit LC-2026-0091, applicant Meridian Import Pte Ltd, beneficiary Alpine Textiles Ltd, amount USD 48,500.00, tolerance plus or minus 5 percent on amount and quantity. Goods: 12,000 units of 100 percent cotton woven fabric, 220gsm, colour navy blue. Latest shipment date 15 September 2026, partial shipments not allowed.",

    "LC-2026-0091 requires the following documents: signed commercial invoice in 3 originals, full set of clean on-board ocean bills of lading consigned to order of issuing bank marked freight prepaid, packing list in 2 originals, certificate of origin issued by Vietnam Chamber of Commerce, and an insurance certificate covering 110 percent of invoice value against all risks. Documents must be presented within 15 days after shipment date but within the credit's validity, which expires 30 September 2026.",

    "Letter of credit LC-2026-0114, applicant Coastal Machinery Trading Co, beneficiary Zhenjiang Precision Parts Co, amount USD 132,000.00, no tolerance clause stated. Goods: 40 units of CNC lathe spare part sets, model PZ-40, new and unused. Latest shipment date 20 September 2026, partial shipments allowed, transshipment not allowed.",

    "LC-2026-0114 requires: signed commercial invoice in 2 originals, full set of clean on-board ocean bills of lading consigned to order of shipper and blank endorsed, packing list in 2 originals, certificate of origin issued by CCPIT, and a beneficiary's certificate confirming one set of non-negotiable documents sent directly to the applicant by courier. Documents must be presented within 10 days after shipment date, credit expires 5 October 2026.",

    "Letter of credit LC-2026-0158, applicant Horizon Foods Distribution Pte Ltd, beneficiary Golden Harvest Agro Exports, amount USD 61,200.00, tolerance plus or minus 10 percent on amount and quantity, described as 'about' 60 metric tons. Goods: basmati rice, grade 1A, in 25kg poly bags. Latest shipment date 25 September 2026, partial shipments allowed.",

    "LC-2026-0158 requires: signed commercial invoice in 3 originals, full set of clean on-board ocean bills of lading consigned to order of issuing bank, packing list in 2 originals, certificate of origin issued by Indian Chamber of Commerce, a phytosanitary certificate issued by a competent authority, and a fumigation certificate. Presentation period is 21 days after shipment date, credit expires 10 October 2026.",

    "UCP 600 Article 14(a) states that a nominated bank, a confirming bank and the issuing bank must examine a presentation to determine, on the basis of the documents alone, whether the documents appear on their face to constitute a complying presentation.",

    "UCP 600 Article 14(d) states that data in a document, when read in context with the credit, other stipulated documents or the document itself, need not be identical to, but must not conflict with, data in that document, any other stipulated document or the credit.",

    "UCP 600 Article 18(c) states that the description of the goods, services or performance in a commercial invoice must correspond with that appearing in the credit.",

    "UCP 600 Article 30(b) allows a tolerance not to exceed 5 percent more or 5 percent less than the quantity of goods, provided the credit does not state the quantity in terms of a stated number of packing units or individual items, and the total drawings do not exceed the credit amount.",

    "UCP 600 Article 31(a) states that partial drawings or shipments are allowed unless the credit stipulates that they are not allowed.",

    "UCP 600 Article 14(c) states that a presentation including one or more original transport documents must be made not later than 21 calendar days after the date of shipment, but in any event not later than the expiry date of the credit.",

    "A past case at this desk: a presentation under LC-2026-0091 was rejected because the bill of lading showed an on-board date of 20 September 2026, five days after the credit's latest shipment date of 15 September 2026. The discrepancy fee charged to the beneficiary was USD 75.",

    "A past case at this desk: a presentation under LC-2026-0114 was flagged because the bill of lading showed one transshipment at Hong Kong, even though the credit explicitly prohibited transshipment. The issuing bank refused the documents and issued a single notice of refusal within the required banking days.",

    "A past case at this desk: a presentation under LC-2026-0158 was accepted even though the invoice quantity was 57 metric tons against a stated 'about 60 metric tons', because the 10 percent tolerance on an 'about' quantity permitted a range of 54 to 66 metric tons.",

    "The examiner's rule of thumb at this desk is that a missing certificate is treated as a discrepancy even if the shipment itself is otherwise compliant, because UCP 600 requires examination of the presentation as a whole, not document by document in isolation.",
]
print(f'{len(DOCS)} documents')
assert 10 <= len(DOCS) <= 40, 'The brief asks for 10-30 documents.'

16 documents


### Chunking
Long documents get cut into overlapping slices before embedding. Why: one embedding holds one
*average* meaning, so a long multi-topic document becomes a blurry vector that matches everything
a little and nothing well. Small chunks keep each vector about a single idea. The **overlap** stops
an idea that straddles a boundary from being cut in half.

This is a knob you tune against your evidence, not a setting with a correct value.

In [4]:
# TRY THIS (knob): watch what changes in Section 3 when you move these.
CHUNK_WORDS = 45     # try 15 (tiny) or 300 (whole document = one chunk)
OVERLAP     = 12     # words shared with the next chunk

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words, out, i = text.split(), [], 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = [(c, d_i) for d_i, d in enumerate(DOCS) for c in chunk(d)]
print(f'{len(DOCS)} docs -> {len(CHUNKS)} chunks (CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP})')

16 docs -> 26 chunks (CHUNK_WORDS=45, OVERLAP=12)


### Embed and retrieve
The embedder runs locally — a small model downloaded once, no key, no cost. If the download fails
(no internet on the machine, or a locked-down environment), the fallback cell below uses **TF-IDF**
word-overlap instead. TF-IDF works, and it is a useful contrast: it matches *words*, not *meaning*,
so it fails on paraphrase. If you use the fallback, say so in your writeup — it changes which
failures you find.

In [5]:
EMBEDDER = 'sentence-transformers/all-MiniLM-L6-v2'
USING = None
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(EMBEDDER)
    def embed(texts):
        v = _m.encode(list(texts), normalize_embeddings=True)
        return np.asarray(v)
    USING = f'local embeddings ({EMBEDDER}) — matches MEANING'
except Exception as e:
    print('embedder unavailable ->', str(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    _v = TfidfVectorizer().fit([c for c, _ in CHUNKS])
    def embed(texts):
        m = _v.transform(list(texts)).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.where(n == 0, 1, n)
    USING = 'TF-IDF fallback — matches WORDS, not meaning'

print('using:', USING)
MATRIX = embed([c for c, _ in CHUNKS])

TOP_K = 3            # TRY THIS (knob): 1 (brittle) or 8 (noisy)

def retrieve(question, k=TOP_K):
    q = embed([question])[0]
    scores = MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], float(scores[i])) for i in order]

for txt, d_i, s in retrieve('how often is the fiddle leaf fig watered in winter?'):
    print(f'  {s:.3f}  [doc {d_i}]  {txt[:80]}...')

c:\Users\小鸮\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\小鸮\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\小鸮\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python 

using: local embeddings (sentence-transformers/all-MiniLM-L6-v2) — matches MEANING
  0.310  [doc 5]  Indian Chamber of Commerce, a phytosanitary certificate issued by a competent au...
  0.261  [doc 0]  woven fabric, 220gsm, colour navy blue. Latest shipment date 15 September 2026, ...
  0.224  [doc 4]  tons. Goods: basmati rice, grade 1A, in 25kg poly bags. Latest shipment date 25 ...


### Answer from the retrieved text — and only from it
The instruction to say *"the notes do not say"* is the whole point. Without it the model fills gaps
from its training data and you cannot tell which sentences came from your documents. That is
**silent failure**: a confident, fluent, unsourced answer.

In [10]:
GROUNDED = (
    "Answer using ONLY the notes provided. "
    "If the notes do not contain the answer, reply exactly: The notes do not say. "
    "Do not use any other knowledge. Quote the phrase you relied on."
)

def answer(question, k=TOP_K, show=True):
    hits = retrieve(question, k)
    context = '\n\n'.join(f'[{i+1}] {t}' for i, (t, _, _) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {question}', system=GROUNDED)
    if show:
        print(f'Q: {question}\nA: {out}\n   (top chunk score {hits[0][2]:.3f}, doc {hits[0][1]})\n')
    return out, hits

answer('What is the amount of LC-2026-0091?')

Q: What is the amount of LC-2026-0091?
A: The notes do not say.
   (top chunk score 0.623, doc 5)



('The notes do not say.',
 [('LC-2026-0158 requires: signed commercial invoice in 3 originals, full set of clean on-board ocean bills of lading consigned to order of issuing bank, packing list in 2 originals, certificate of origin issued by Indian Chamber of Commerce, a phytosanitary certificate issued by a competent authority,',
   5,
   0.6226564049720764),
  ("LC-2026-0114 requires: signed commercial invoice in 2 originals, full set of clean on-board ocean bills of lading consigned to order of shipper and blank endorsed, packing list in 2 originals, certificate of origin issued by CCPIT, and a beneficiary's certificate confirming one set of non-negotiable",
   3,
   0.6111186742782593),
  ('LC-2026-0091 requires the following documents: signed commercial invoice in 3 originals, full set of clean on-board ocean bills of lading consigned to order of issuing bank marked freight prepaid, packing list in 2 originals, certificate of origin issued by Vietnam Chamber of Commerce, and an',
 

---
## 2 · Five questions it could not answer without your documents  ### REPLACE ME

Each question should have a **specific** answer that lives in your documents — a date, a name, a
number, a decision and its outcome. "What is a monstera?" is a bad question: the model knows that
already, so a correct answer proves nothing about your retrieval.

In [11]:
### REPLACE ME — five questions answerable ONLY from your documents.
QUESTIONS = [
      "What colour and weight in gsm is the fabric specified in LC-2026-0091, and what is the beneficiary company's name?",
    "What made the invoice quantity acceptable under LC-2026-0158's case study even though it differed from the stated 'about 60 metric tons'?",
    "Why was the LC-2026-0114 transshipment case rejected, and what did the issuing bank do about it?",
    "How many days after the shipment date must documents be presented under LC-2026-0158, and when does that credit expire?",
    "According to the examiner's rule of thumb, is a single missing certificate enough to flag a discrepancy even if everything else is compliant?",
]
for q in QUESTIONS:
    answer(q)

Q: What colour and weight in gsm is the fabric specified in LC-2026-0091, and what is the beneficiary company's name?
A: The colour is navy blue and the weight is 220gsm. The beneficiary company's name is Alpine Textiles Ltd. 

This is based on the phrases: "colour navy blue" and "beneficiary Alpine Textiles Ltd."
   (top chunk score 0.552, doc 0)

Q: What made the invoice quantity acceptable under LC-2026-0158's case study even though it differed from the stated 'about 60 metric tons'?
A: The invoice quantity was acceptable under LC-2026-0158's case study because "the 10 percent tolerance on an 'about' quantity permitted a range of 54 to 66 metric tons."
   (top chunk score 0.835, doc 14)

Q: Why was the LC-2026-0114 transshipment case rejected, and what did the issuing bank do about it?
A: The LC-2026-0114 transshipment case was rejected because "the bill of lading showed one transshipment at Hong Kong, even though the credit explicitly prohibited transshipment." The issuing bank "re

---
## 3 · Break it  ### REPLACE ME

**This section carries the marks.** You are looking for a question where retrieval brings back the
wrong passage — not where the model writes a clumsy sentence.

Four kinds of break that reliably work. Try all four; keep the one that fails most interestingly.

| Kind | What to try |
|---|---|
| **Vocabulary mismatch** | Ask using a word that never appears in your documents but means the same thing. Devastating for TF-IDF; embeddings usually survive it. |
| **Two documents needed** | Ask something that requires combining facts from two documents. Top-k retrieval fetches the *most similar* chunks, which is not the same as *the set that answers the question*. |
| **Distractor** | Ask about a topic covered in several documents where only one is relevant. Watch the right chunk get outvoted. |
| **Absent but plausible** | Ask something your documents nearly, but do not quite, cover. Does it say "the notes do not say", or does it invent? |

**Diagnose, do not just report.** "It got it wrong" is a Developing answer. "The right chunk ranked
fourth with score 0.31 while an irrelevant chunk scored 0.44 because both contain the word
*water*" is a Proficient one.

In [12]:
### REPLACE ME — your breaking attempts.
BREAKERS = [
     ("vocabulary mismatch", "For the machinery parts credit, is sending the goods via a third port before final destination permitted?"),
    ("two documents needed", "Which letter of credit has the strictest presentation period, and how many days shorter is it than the rice shipment's period?"),
    ("distractor", "Which certificate of origin was issued by CCPIT?"),
    ("absent but plausible", "What is the discrepancy fee for a late presentation under LC-2026-0158?"),
]
for kind, q in BREAKERS:
    print(f'--- {kind.upper()} ---')
    out, hits = answer(q)
    print('   retrieved:')
    for t, d_i, s in hits:
        print(f'     {s:.3f} [doc {d_i}] {t[:70]}...')
    print()

--- VOCABULARY MISMATCH ---
Q: For the machinery parts credit, is sending the goods via a third port before final destination permitted?
A: The notes do not say.
   (top chunk score 0.448, doc 2)

   retrieved:
     0.448 [doc 2] Letter of credit LC-2026-0114, applicant Coastal Machinery Trading Co,...
     0.426 [doc 10] UCP 600 Article 31(a) states that partial drawings or shipments are al...
     0.399 [doc 2] and unused. Latest shipment date 20 September 2026, partial shipments ...

--- TWO DOCUMENTS NEEDED ---
Q: Which letter of credit has the strictest presentation period, and how many days shorter is it than the rice shipment's period?
A: The notes do not say.
   (top chunk score 0.594, doc 5)

   retrieved:
     0.594 [doc 5] Indian Chamber of Commerce, a phytosanitary certificate issued by a co...
     0.515 [doc 11] UCP 600 Article 14(c) states that a presentation including one or more...
     0.489 [doc 3] issued by CCPIT, and a beneficiary's certificate confirming one set o

### Diagnose the one you will submit
Fill this in **in prose**, from the scores printed above. This is your documented failure.

In [13]:
FAILURE_DIAGNOSIS = """
Question that failed: "How many days after the shipment date must documents be
presented under LC-2026-0158, and when does that credit expire?"

What the system answered: "Documents must be presented within 10 days after shipment
date, and the credit expires 5 October 2026." Stated confidently, with no hedge.

What the correct answer was: LC-2026-0158 requires presentation within 21 days after
shipment date, and the credit expires 10 October 2026.

Which chunk SHOULD have been retrieved, and what did it score:
The chunk from doc 5 describing LC-2026-0158's own presentation period ("Presentation
period is 21 days after shipment date, credit expires 10 October 2026") was not among
the top-3 retrieved chunks for this query.

What was retrieved instead, and what did it score:
The top chunk (score 0.796) was from doc 3, which states LC-2026-0114's terms ("within
10 days after shipment date... credit expires 5 October 2026") — a different letter of
credit entirely, but one with the same sentence structure and field type (a presentation
period and an expiry date).

WHY (the mechanism): doc 3 and doc 5 both describe "presentation period... days after
shipment date" and "expires [date]" using near-identical phrasing, so their embeddings
sit close together in vector space even though the numbers they contain belong to
different LCs. Retrieval picked the structurally similar but factually wrong document
as the top match — a distractor failure. The more serious problem sits in generation,
not retrieval: the grounding instruction says to decline when the notes do not cover
the question, but the model instead answered fluently and confidently using the wrong
LC's numbers, producing exactly the silent, unsourced-seeming failure the FAQ warns
about — a wrong answer that looks like a right one.

What you would change to fix it, and what that change would cost you elsewhere:
Including the LC identifier (e.g. "LC-2026-0158") inside each chunk's text (not just as
metadata) would let embeddings distinguish "LC-2026-0158's period" from "LC-2026-0114's
period" even when the surrounding sentence structure is nearly identical. The cost:
repeating the LC number in every chunk increases each chunk's token count and dilutes
the semantic signal that is not the identifier, which could weaken retrieval for
identifier-free questions. A cheaper partial fix is a stricter grounding instruction
that requires the model to quote the specific LC number from the retrieved chunk before
answering, so a mismatch between the question's LC and the chunk's LC becomes visible
rather than silently substituted.
"""
print(FAILURE_DIAGNOSIS)


Question that failed: "How many days after the shipment date must documents be
presented under LC-2026-0158, and when does that credit expire?"

What the system answered: "Documents must be presented within 10 days after shipment
date, and the credit expires 5 October 2026." Stated confidently, with no hedge.

What the correct answer was: LC-2026-0158 requires presentation within 21 days after
shipment date, and the credit expires 10 October 2026.

Which chunk SHOULD have been retrieved, and what did it score:
The chunk from doc 5 describing LC-2026-0158's own presentation period ("Presentation
period is 21 days after shipment date, credit expires 10 October 2026") was not among
the top-3 retrieved chunks for this query.

What was retrieved instead, and what did it score:
The top chunk (score 0.796) was from doc 3, which states LC-2026-0114's terms ("within
10 days after shipment date... credit expires 5 October 2026") — a different letter of
credit entirely, but one with the same sen

---
## 4 · Two checks worth running before you write

**Check 1 — is retrieval doing anything?** Ask the same five questions with the documents removed.
If the model still answers correctly, your documents are too generic and Part 2 has not been
demonstrated. Fix the documents.

**Check 2 — does the grounding instruction hold?** A system that invents when it should decline is
worse than one that declines too often, because you cannot see it happening.

In [14]:
print('=== CHECK 1: same questions, NO documents ===')
for q in QUESTIONS[:3]:
    print(f'Q: {q}\nA: {generate(q, system="Answer in one short sentence.", max_new_tokens=80)}\n')

print('=== CHECK 2: something your documents definitely do not cover ===')
answer('What is the postal code of the flat?')

=== CHECK 1: same questions, NO documents ===
Q: What colour and weight in gsm is the fabric specified in LC-2026-0091, and what is the beneficiary company's name?
A: I'm sorry, but I don't have access to specific data or documents such as LC-2026-0091.

Q: What made the invoice quantity acceptable under LC-2026-0158's case study even though it differed from the stated 'about 60 metric tons'?
A: The invoice quantity was deemed acceptable because it fell within the permissible tolerance range specified in the letter of credit.

Q: Why was the LC-2026-0114 transshipment case rejected, and what did the issuing bank do about it?
A: The LC-2026-0114 transshipment case was rejected due to non-compliance with the terms of the letter of credit, prompting the issuing bank to refuse payment.

=== CHECK 2: something your documents definitely do not cover ===
Q: What is the postal code of the flat?
A: The notes do not say.
   (top chunk score 0.313, doc 1)



('The notes do not say.',
 [('LC-2026-0091 requires the following documents: signed commercial invoice in 3 originals, full set of clean on-board ocean bills of lading consigned to order of issuing bank marked freight prepaid, packing list in 2 originals, certificate of origin issued by Vietnam Chamber of Commerce, and an',
   1,
   0.313393771648407),
  ('LC-2026-0158 requires: signed commercial invoice in 3 originals, full set of clean on-board ocean bills of lading consigned to order of issuing bank, packing list in 2 originals, certificate of origin issued by Indian Chamber of Commerce, a phytosanitary certificate issued by a competent authority,',
   5,
   0.30275991559028625),
  ('Letter of credit LC-2026-0091, applicant Meridian Import Pte Ltd, beneficiary Alpine Textiles Ltd, amount USD 48,500.00, tolerance plus or minus 5 percent on amount and quantity. Goods: 12,000 units of 100 percent cotton woven fabric, 220gsm, colour navy blue. Latest shipment date 15 September 2026,',
 

---
## 5 · YOUR TURN

1. Replace `DOCS` with 10–30 documents from your domain that contain things a model cannot know.
2. Replace `QUESTIONS` with five that are answerable only from them. Run and read every answer.
3. Run Check 1. If the model answers without your documents, go back to step 1.
4. Work through all four `BREAKERS`. Keep the most interesting failure and diagnose it properly.
5. Try `CHUNK_WORDS`, `OVERLAP` and `TOP_K` at extremes. Note which failures appear and disappear.

### What your ~400 words must contain

- What your documents are, and why a model could not answer these questions without them.
- The five worked examples (the notebook output is enough; the words are for what they show).
- **The failure, diagnosed** — which chunk should have won, which did, the scores, and the
  mechanism.
- What you would change to fix it, and **what that fix would break** — bigger chunks improve
  context and dilute matching; a larger top-k improves recall and adds noise and tokens. There is
  no free setting, and saying so is the postgraduate move.

In [ ]:
print(f'tokens this session: {TOKENS["in"]} in / {TOKENS["out"]} out')

tokens this session: 2878 in / 242 out
